In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [ ]:
username = 
password = 
auth = (username, password)

In [5]:
data_element_anc = pd.read_excel("../data/metadata/anc_data_element_required.xlsx")

In [15]:
data_element_anc[data_element_anc['priotised'] == 1]

,hmis_data_element_id,hmis_data_element_name,priotised
0,A2JDrKjR3nr,ANC First standard contact 1st trimester,1
1,A5HjXlj1bhf,ANC new registrations syphilis tested positive,1
2,CIGr10eQg8N,ANC new registrations received ultrasound scan,1
3,D702xSW9bHo,ANC new registrations screened for malnutritio...,1
4,DRQdxGkLtHC,ANC new registrations under 14 years,1
6,FWuzAt99w1y,ANC risk pregnancy detected_including pregnanc...,1
7,FqDAfzXHRJo,ANC new registrations tested for syphilis,1
8,K0nNhsjxzbR,ANC new registrations with anemia Hb < 11 g/dL,1
10,MTe1TWLuUAv,ANC new registrations screened who were malnou...,1
12,Qtei9Sx1QpS,ANC Number of pregnant women with HBV infectio...,1


In [6]:
df_org = pd.read_excel('../data/metadata/hmis_organizations.xlsx')

In [16]:
# 2) Fixed period (full year) — adjust as needed
start_date = "2025-04-01"
end_date   = "2026-04-20"

all_records = []

url = "https://aggregate.moh.gov.rw/api/dataValueSets"

# 3) Loop over data elements (one DE per request)
for de in data_element_anc[data_element_anc['priotised'] == 1]['hmis_data_element_id'].dropna().to_list():
    print(f"Fetching DE {de} for {start_date} → {end_date}...")
    params = {
        "dataElement": de,         # single DE per request
        "startDate": start_date,
        "endDate": end_date,
        "orgUnit": "Hjw70Lodtf2",
        "children": "true"         # DHIS2 expects "true"/"false"
    }

    resp = requests.get(url, params=params, auth=auth)
    if resp.status_code == 200:
        rows = resp.json().get("dataValues", [])
        if rows:
            all_records.extend(rows)
    else:
        print(f"Error {resp.status_code}: {resp.text[:300]}")

# 4) Combine into DataFrame (+ optional cleanup)
df = pd.DataFrame(all_records)

print(f"Total rows fetched: {len(df)}")

Fetching DE A2JDrKjR3nr for 2025-04-01 → 2026-04-20...
Fetching DE A5HjXlj1bhf for 2025-04-01 → 2026-04-20...
Fetching DE CIGr10eQg8N for 2025-04-01 → 2026-04-20...
Fetching DE D702xSW9bHo for 2025-04-01 → 2026-04-20...
Fetching DE DRQdxGkLtHC for 2025-04-01 → 2026-04-20...
Fetching DE FWuzAt99w1y for 2025-04-01 → 2026-04-20...
Fetching DE FqDAfzXHRJo for 2025-04-01 → 2026-04-20...
Fetching DE K0nNhsjxzbR for 2025-04-01 → 2026-04-20...
Fetching DE MTe1TWLuUAv for 2025-04-01 → 2026-04-20...
Fetching DE Qtei9Sx1QpS for 2025-04-01 → 2026-04-20...
Fetching DE VNuJKau0K2w for 2025-04-01 → 2026-04-20...
Fetching DE XycRcoG0nnz for 2025-04-01 → 2026-04-20...
Fetching DE Y42lqkBSLyA for 2025-04-01 → 2026-04-20...
Fetching DE Y4xlw0XUzhR for 2025-04-01 → 2026-04-20...
Fetching DE fEx6ze7m0k8 for 2025-04-01 → 2026-04-20...
Fetching DE ri0XrmXSpEC for 2025-04-01 → 2026-04-20...
Fetching DE viftAbwOnMH for 2025-04-01 → 2026-04-20...
Fetching DE zajzl4FQFoI for 2025-04-01 → 2026-04-20...
Total rows

In [ ]:
# df.to_csv("hmis_data/obs_complications/obs_hc_sep_2025_mar2026.csv")

In [ ]:
# df = pd.read_csv("hmis_data/obs_complications/obs_hc_sep_2025_mar2026.csv")
# df = df.drop(columns = 'Unnamed: 0')

C:\Users\Muzungu Hirwa\AppData\Local\Temp\ipykernel_12996\4254214101.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("hmis_data/obs_complications/obs_hc_sep_2025_mar2026.csv")


In [17]:
df_1 = df.copy()

#### Adding Key Metadata
- Data ELement Names
- Category Option COmbinations
- Organization

In [18]:
url = "https://aggregate.moh.gov.rw/api/dataElements"

params = {
    "fields": "id,name",
    "paging": "false"
}

response = requests.get(url, params=params, auth=auth)
data = response.json()["dataElements"]

data_element_meta = pd.DataFrame(data)

In [19]:
data_element_name_mapping = dict(zip(data_element_meta['id'], data_element_meta['name']))

# Insert after dataElement
de_idx = df_1.columns.get_loc('dataElement')
df_1.insert(de_idx + 1, 'data_element_name', df_1['dataElement'].map(data_element_name_mapping))

In [20]:
url = "https://aggregate.moh.gov.rw/api/categoryOptionCombos"

params = {
    "fields": "id,name",
    "paging": "false"
}

response = requests.get(url, params=params, auth=auth)
data = response.json()["categoryOptionCombos"]

category_options = pd.DataFrame(data)

In [21]:
category_options_mapping = dict(zip(category_options['id'], category_options['name']))

# Insert after dataElement
de_idx = df_1.columns.get_loc('categoryOptionCombo')
df_1.insert(de_idx + 1, 'category_option_combo', df_1['categoryOptionCombo'].map(category_options_mapping))

In [22]:
df_2 = df_1[[ 'period', 'orgUnit','dataElement', 'data_element_name','category_option_combo', 'value']]

#### Adding Organizations

In [23]:
df_3 = pd.merge(
    df_2, df_org[['hmis_fac_id','fosa_code', 'hmis_fac_name', 'Province', 'District', 'Subdistrict', 'Sector', "facility_type", "facility_category"]],  # removed 'id' to avoid id_x/id_y conflict
    left_on='orgUnit',
    right_on='hmis_fac_id',
    how='left'
)

### Adding Period

In [24]:
df_3['period'] = pd.to_datetime(df_3['period'].astype(str), format='%Y%m').dt.to_period('M')

In [25]:
df_3

,period,orgUnit,dataElement,data_element_name,category_option_combo,value,hmis_fac_id,fosa_code,hmis_fac_name,Province,District,Subdistrict,Sector,facility_type,facility_category
0,2025-04,buguOgodg9Y,A2JDrKjR3nr,ANC First standard contact 1st trimester,default,13,buguOgodg9Y,573,Ngange CS,West,Nyamasheke,Kibogora Sub District,Karambi,Public,Health Center
1,2025-04,Qp6EM25AmgR,A2JDrKjR3nr,ANC First standard contact 1st trimester,default,42,Qp6EM25AmgR,94,Rusatira-kinazi CS,South,Huye,Kabutare Sub District,Rusatira,Public,Health Center
2,2025-04,L1ejzhqcmOo,A2JDrKjR3nr,ANC First standard contact 1st trimester,default,40,L1ejzhqcmOo,433,Gituku CS,East,Ngoma,Kibungo Sub District,Rukira,Public,Health Center
3,2025-04,UNpU4eA39Nv,A2JDrKjR3nr,ANC First standard contact 1st trimester,default,31,UNpU4eA39Nv,96,Simbi CS,South,Huye,Kabutare Sub District,Simbi,Public,Health Center
4,2025-04,F6yOxGNU4hF,A2JDrKjR3nr,ANC First standard contact 1st trimester,default,33,F6yOxGNU4hF,266,Mwezi CS,West,Nyamasheke,Bushenge Sub District,Karengera,Public,Health Center
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133853,2026-03,K5xVzrHnSNb,zajzl4FQFoI,ANC Insecticide Treated Bed nets distributed,default,0,K5xVzrHnSNb,3441,Buliba SGHP,East,Ngoma,Kibungo Sub District,Rukira,Unknown,Health Post 2nd Generation
133854,2026-03,XEFWoDU1oTt,zajzl4FQFoI,ANC Insecticide Treated Bed nets distributed,default,0,XEFWoDU1oTt,3442,Kamatamu I HP,Kigali City,Gasabo,Kibagabaga Sub District,Kacyiru,Unknown,Health Post
133855,2026-03,C0hF02NIkKo,zajzl4FQFoI,ANC Insecticide Treated Bed nets distributed,default,0,C0hF02NIkKo,3457,Saint Vincent HP,Kigali City,Gasabo,Kibagabaga Sub District,Kinyinya,Unknown,Health Post
133856,2026-03,rvRYfDqmPA7,zajzl4FQFoI,ANC Insecticide Treated Bed nets distributed,default,0,rvRYfDqmPA7,3473,Nyagihunika SGHP,East,Bugesera,Nyamata Sub District,Musenyi,Unknown,Other


In [28]:
df_3.to_excel("anc_hmis_apri_2025_mar_2026.xlsx", index=False)

In [72]:
df_3['period'].value_counts()

period
2026-03    40708
2026-01    40554
2025-12    40298
2026-02    40130
2025-11    37567
2025-10    37306
2025-09    37061
Freq: M, Name: count, dtype: int64